In [0]:
%pip install xgboost optuna shap
dbutils.library.restartPython()

In [0]:
import mlflow
import xgboost as xgb
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from mlflow.models import infer_signature
from mlflow import MlflowClient
import shap
import matplotlib.pyplot as plt

catalog = "credit_risk_fraud_detection"
mlflow.set_registry_uri("databricks-uc")


In [0]:
display(spark.sql(f"""
    select is_fraud, count(*) as n
    from {catalog}.silver.fct_transactions
    group by is_fraud
"""))

In [0]:
# MAGIC %md
# MAGIC ## Temporal split at the Spark level, THEN sample
# MAGIC Split by transaction_date first (train on earlier days, test on later
# MAGIC days) so no temporal leakage, then sample negatives within each window
# MAGIC separately -- heavy undersampling for train (learnable + fast), a much
# MAGIC larger sample for test (more realistic precision estimate).

In [0]:
from pyspark.sql import functions as F

split_date = spark.sql(f"""
    select percentile_approx(unix_timestamp(transaction_date), 0.85) as split_ts
    from {catalog}.silver.fct_transactions
""").collect()[0]["split_ts"]

fct = spark.table(f"{catalog}.silver.fct_transactions")

train_window = fct.filter(F.unix_timestamp("transaction_date") <= split_date)
test_window  = fct.filter(F.unix_timestamp("transaction_date") >  split_date)

# Training set: all positives + undersampled negatives (target ~1:50 ratio)
train_pos = train_window.filter(F.col("is_fraud") == True)
train_pos_count = train_pos.count()
train_neg_target = train_pos_count * 50
train_neg_total = train_window.filter(F.col("is_fraud") == False).count()
train_neg_frac = min(1.0, train_neg_target / train_neg_total)
train_neg = train_window.filter(F.col("is_fraud") == False).sample(fraction=train_neg_frac, seed=42)

train_df = train_pos.unionByName(train_neg)

# Test set: all positives + a larger, less-distorted negative sample (up to 500k)
test_pos = test_window.filter(F.col("is_fraud") == True)
test_neg_total = test_window.filter(F.col("is_fraud") == False).count()
test_neg_frac = min(1.0, 500_000 / test_neg_total)
test_neg = test_window.filter(F.col("is_fraud") == False).sample(fraction=test_neg_frac, seed=42)

test_df = test_pos.unionByName(test_neg)

print(f"train: {train_df.count()} rows ({train_pos_count} positive)")
print(f"test:  {test_df.count()} rows ({test_pos.count()} positive)")

train_pdf = train_df.toPandas()
test_pdf = test_df.toPandas()

In [0]:
# MAGIC %md ## Feature prep

In [0]:
categorical_cols = ["transaction_type", "transaction_mode", "channel"]

numeric_cols = [
    "amount", "distance_from_home_km",
    "velocity_1hr", "velocity_24hr", "amount_vs_30d_avg_ratio", "time_since_last_txn_mins",
]

bool_cols = ["is_new_device", "is_new_merchant_category", "is_night_transaction"]

for df_ in (train_pdf, test_pdf):
    for c in numeric_cols:
        df_[c] = pd.to_numeric(df_[c], errors="coerce")
    for c in bool_cols:
        df_[c] = df_[c].astype(bool)

# Categorical columns: derive the category set from the UNION of train+test,
# then apply that SAME set to both -- prevents XGBoost seeing a category at
# test time that didn't exist in train's category levels.
for c in categorical_cols:
    combined_categories = pd.concat([train_pdf[c], test_pdf[c]]).astype("category").cat.categories
    train_pdf[c] = pd.Categorical(train_pdf[c], categories=combined_categories)
    test_pdf[c] = pd.Categorical(test_pdf[c], categories=combined_categories)

feature_cols = numeric_cols + bool_cols + categorical_cols
target_col = "is_fraud"

X_train, y_train = train_pdf[feature_cols], train_pdf[target_col].astype(int)
X_test, y_test = test_pdf[feature_cols], test_pdf[target_col].astype(int)

In [0]:
# MAGIC %md ## Optuna hyperparameter search (50 trials)

In [0]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "enable_categorical": True,
        "tree_method": "hist",
        "scale_pos_weight": scale_pos_weight,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, preds)   # optimize PR-AUC, not ROC-AUC, given imbalance

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best PR-AUC:", study.best_value)
print("Best params:", study.best_params)

In [0]:
# MAGIC %md ## Train final model, log to MLflow, register in Unity Catalog

In [0]:
def ks_statistic(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return max(tpr - fpr)

with mlflow.start_run(run_name="xgboost_fraud_classifier") as run:
    best_params = dict(study.best_params)
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "enable_categorical": True,
        "tree_method": "hist",
        "scale_pos_weight": scale_pos_weight,
    })

    final_model = xgb.XGBClassifier(**best_params)
    final_model.fit(X_train, y_train)

    test_preds = final_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_preds)
    test_ks = ks_statistic(y_test, test_preds)
    test_pr_auc = average_precision_score(y_test, test_preds)

    mlflow.log_params(best_params)
    mlflow.log_metric("test_roc_auc", test_auc)
    mlflow.log_metric("test_ks_statistic", test_ks)
    mlflow.log_metric("test_pr_auc", test_pr_auc)
    mlflow.log_metric("train_positive_count", int(train_pos_count))
    mlflow.log_metric("test_positive_count", int(test_pos.count()))

    print(f"Test ROC-AUC: {test_auc:.4f}")
    print(f"Test KS:      {test_ks:.4f}")
    print(f"Test PR-AUC:  {test_pr_auc:.4f}")

    # SHAP
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, show=False)
    plt.savefig("/tmp/shap_summary_fraud.png", bbox_inches="tight")
    mlflow.log_artifact("/tmp/shap_summary_fraud.png")
    plt.close()

    from sklearn.metrics import PrecisionRecallDisplay
    PrecisionRecallDisplay.from_predictions(y_test, test_preds)
    plt.savefig("/tmp/pr_curve_fraud.png", bbox_inches="tight")
    mlflow.log_artifact("/tmp/pr_curve_fraud.png")
    plt.close()

    signature = infer_signature(X_test, final_model.predict(X_test))
    mlflow.xgboost.log_model(
        final_model,
        artifact_path="model",
        registered_model_name=f"{catalog}.ml.fraud_classifier_model",
        signature=signature,
        input_example=X_test.head(5),
    )

    run_id = run.info.run_id

print("Run ID:", run_id)


In [0]:
# MAGIC %md ## Set @champion alias

In [0]:
client = MlflowClient()
model_name = f"{catalog}.ml.fraud_classifier_model"
versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(int(v.version) for v in versions)
client.set_registered_model_alias(model_name, "champion", latest_version)
print(f"Set @champion alias -> version {latest_version}")